In [13]:
!pip install plyfile open3d numpy

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple

[notice] A new release of pip is available: 23.3.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 物体A（纸巾）.ply转为.obj

In [14]:
import subprocess

script_path = "/mnt/workspace/process_tissue_clean.py"

with open(script_path, "w", encoding="utf-8") as f:
    f.write('''import numpy as np

# 🔥 强行修复 NumPy 1.24+ 移除 np.long 的问题
if not hasattr(np, 'long'):
    np.long = int

import open3d as o3d
from plyfile import PlyData

# ================= 配置路径 =================
input_ply_path = "/mnt/workspace/data/tissue.ply"
output_obj_path = "/mnt/workspace/data/tissue.obj"
# ============================================

print("📦 1. 正在深度解析高斯点云资产...")
plydata = PlyData.read(input_ply_path)
vertex_data = plydata['vertex']

xyz = np.stack([vertex_data['x'], vertex_data['y'], vertex_data['z']], axis=-1)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)

print("📐 2. 正在处理与定向法向量...")
raw_normals = np.stack([vertex_data['nx'], vertex_data['ny'], vertex_data['nz']], axis=-1)
normal_lengths = np.linalg.norm(raw_normals, axis=-1)

if np.allclose(normal_lengths, 0.0) or np.isnan(raw_normals).any():
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamKNN(knn=30))
    pcd.orient_normals_consistent_tangent_plane(100)
else:
    normal_lengths[normal_lengths == 0] = 1.0
    normalized_normals = raw_normals / normal_lengths[:, np.newaxis]
    pcd.normals = o3d.utility.Vector3dVector(normalized_normals)
    pcd.orient_normals_consistent_tangent_plane(30)

print("🎨 3. 正在解算 3DGS 球谐系数并烧录真彩色...")
f_dc_0 = vertex_data['f_dc_0']
f_dc_1 = vertex_data['f_dc_1']
f_dc_2 = vertex_data['f_dc_2']
C0 = 0.28209479177387814

prop_names = [p.name for p in vertex_data.properties]
f_rest_0 = vertex_data['f_rest_0'] if 'f_rest_0' in prop_names else 0
f_rest_1 = vertex_data['f_rest_1'] if 'f_rest_1' in prop_names else 0
f_rest_2 = vertex_data['f_rest_2'] if 'f_rest_2' in prop_names else 0

rgb = np.stack([
    0.5 + f_dc_0 * C0 + 0.1 * f_rest_0, 
    0.5 + f_dc_1 * C0 + 0.1 * f_rest_1, 
    0.5 + f_dc_2 * C0 + 0.1 * f_rest_2
], axis=-1)
pcd.colors = o3d.utility.Vector3dVector(np.clip(rgb, 0.0, 1.0))

print("⚡ 4. 正在执行泊松曲面重建...")
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
densities = np.asarray(densities)

if len(densities) == 0:
    print("❌ 错误：泊松重建未能生成有效网格。")
else:
    print("✂️ 5. 正在剔除低密度边缘...")
    vertices_to_remove = densities < np.quantile(densities, 0.05)
    mesh.remove_vertices_by_mask(vertices_to_remove)

    # =======================================================
    # 🎯 核心新增：清理分离的碎块（提取最大连通图）
    # =======================================================
    print("🧹 6. 正在扫描并清理分离的浮空碎块...")
    # 把网格按连通性分成不同的聚类
    triangle_clusters, cluster_n_triangles, cluster_area = mesh.cluster_connected_triangles()
    triangle_clusters = np.asarray(triangle_clusters)
    cluster_n_triangles = np.asarray(cluster_n_triangles)

    if len(cluster_n_triangles) > 1:
        print(f"   - 🔎 侦测到 {len(cluster_n_triangles)} 个不相连的网格块！")
        # 找到拥有最多面的那个“主物体”的索引
        largest_cluster_idx = cluster_n_triangles.argmax()
        
        # 制作一个遮罩：如果面不属于主物体，就标记为 True（准备删除）
        triangles_to_remove = triangle_clusters != largest_cluster_idx
        
        # 剔除所有分离的碎块面
        mesh.remove_triangles_by_mask(triangles_to_remove)
        
        # 顺便清理掉那些因为面被删掉而孤立出来的冗余顶点
        mesh.remove_unreferenced_vertices()
        print("   - 🗑️ 较小的游离碎块已全部成功剔除，仅保留主物体。")
    else:
        print("   - 🟢 模型非常完整，未发现需要清理的游离碎块。")

    print(f"💾 7. 正在导出最终纯净版曲面网格 (Mesh)...")
    o3d.io.write_triangle_mesh(output_obj_path, mesh)
    print(f"\\n🎉【大功告成】纯净版 Tissue 资产已生成！\\n👉 路径: {output_obj_path}")
''')

# 唤醒后台 Python 执行
print("🚀 正在启动独立进程进行模型重建与碎块清理...\n")
cmd = ["python", script_path]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")

process.wait()
print("\n✅ 清理完毕，赶紧去看看新模型吧！")

🚀 正在启动独立进程进行模型重建与碎块清理...

📦 1. 正在深度解析高斯点云资产...
📐 2. 正在处理与定向法向量...
🎨 3. 正在解算 3DGS 球谐系数并烧录真彩色...
⚡ 4. 正在执行泊松曲面重建...
✂️ 5. 正在剔除低密度边缘...
🧹 6. 正在扫描并清理分离的浮空碎块...
   - 🔎 侦测到 34 个不相连的网格块！
   - 🗑️ 较小的游离碎块已全部成功剔除，仅保留主物体。
💾 7. 正在导出最终纯净版曲面网格 (Mesh)...

🎉【大功告成】纯净版 Tissue 资产已生成！
👉 路径: /mnt/workspace/data/tissue.obj

✅ 清理完毕，赶紧去看看新模型吧！


## 修正1

In [ ]:
import subprocess

# 升级脚本版本号至 v3，确保节点兼容性
script_path = "/mnt/workspace/pipeline_render_colored_video_v3.py"

with open(script_path, "w") as f:
    f.write('''import bpy
import math
import mathutils
import os

print("🚀 启动【Blender 4.0 节点适配】36帧全彩环绕视频管线...")

# 一、初始化与彻底清空
bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete(use_global=False)

scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.device = 'CPU'
scene.cycles.samples = 12  # 保持短平快低采样，确保快速出图

scene.render.resolution_x = 1280
scene.render.resolution_y = 720

FRAME_COUNT = 36
scene.frame_start = 1
scene.frame_end = FRAME_COUNT

# 二、🛠️ 核心函数 1：强行绑定图片贴图（用于水果 B 和 C）
def force_bind_texture(obj_name, texture_path):
    obj = bpy.data.objects.get(obj_name)
    if not obj or not os.path.exists(texture_path):
        return
    mat = bpy.data.materials.new(name=f"Mat_{obj_name}")
    mat.use_nodes = True
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    bsdf = nodes.get("Principled BSDF")
    
    tex_image = nodes.new('ShaderNodeTexImage')
    tex_image.image = bpy.data.images.load(texture_path)
    links.new(tex_image.outputs['Color'], bsdf.inputs['Base Color'])
    
    if len(obj.data.materials) == 0: obj.data.materials.append(mat)
    else: obj.data.materials[0] = mat

# 三、🛠️ 核心函数 2：强行唤醒点云“顶点颜色”（💡 针对 Blender 4.0 深度优化）
def awake_vertex_color(obj_name):
    obj = bpy.data.objects.get(obj_name)
    if not obj: return
    
    # 检查网格是否自带点云颜色属性
    if not obj.data.color_attributes:
        print(f"ℹ️ 背景网格 {obj_name} 未检测到嵌入的顶点颜色，将保持默认材质。")
        return
        
    active_attr = obj.data.color_attributes.active
    print(f"🎨 ✨检测到点云颜色属性: '{active_attr.name}'，正在适配 Blender 4.0 属性节点树...")
    
    mat = bpy.data.materials.new(name=f"Mat_VC_{obj_name}")
    mat.use_nodes = True
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    bsdf = nodes.get("Principled BSDF")
    
    # 🔥 核心修正：Blender 4.0 必须使用 'ShaderNodeAttribute' 节点来抓取顶点颜色
    attr_node = nodes.new('ShaderNodeAttribute')
    attr_node.attribute_name = active_attr.name
    
    # 将属性节点的 Color 输出连入原理化 BSDF 的 Base Color
    links.new(attr_node.outputs['Color'], bsdf.inputs['Base Color'])
    
    if len(obj.data.materials) == 0: obj.data.materials.append(mat)
    else: obj.data.materials[0] = mat

# 四、导入并部署资产（沿用您的黄金绝对坐标）
layout_data = {
    "TABLE": {
        "path": "/mnt/workspace/data/counter_surface_mesh_poisson.obj",
        "loc": [0.0, 0.0, 0.0],
        "rot": [0.9948, 0.0, 0.0]
    },
    "Obj_B": {
        "path": "/mnt/workspace/data/obj_b.obj",
        "tex": "/mnt/workspace/data/obj_b_text.jpg", 
        "loc": [3.0103, 0.5386, 3.1579],
        "rot": [0.1063, 0.0618, -0.1891]
    },
    "Obj_C": {
        "path": "/mnt/workspace/data/obj_c.obj",
        "tex": "/mnt/workspace/data/obj_c_text.jpg",
        "loc": [1.0591, 0.4646, 2.4416],
        "rot": [1.5707, -1.5910, 0.0]
    }
}

for item_name, info in layout_data.items():
    if not os.path.exists(info["path"]): continue
    bpy.ops.wm.obj_import(filepath=info["path"], forward_axis='Y', up_axis='Z')
    obj = bpy.context.selected_objects[0]
    obj.name = item_name
    obj.location = info["loc"]
    obj.rotation_euler = info["rot"]
    
    # 分流处理材质
    if item_name == "TABLE":
        awake_vertex_color(item_name) # 适配 4.0 的点云着色引擎
    elif "tex" in info:
        force_bind_texture(item_name, info["tex"]) # 强绑水果贴图

# 五、环境打光
bpy.ops.object.light_add(type='SUN', location=(-5, 5, 12))
sun = bpy.context.object
sun.data.energy = 6.0

# 六、创建并配置摄像机
cam_data = bpy.data.cameras.new("cam")
cam_data.lens = 20  
cam = bpy.data.objects.new("cam", cam_data)
bpy.context.collection.objects.link(cam)
scene.camera = cam

# 🎯 保持上一节你要求的：转轴向物体 B 靠近 60%
pos_b = mathutils.Vector(layout_data["Obj_B"]["loc"])
pos_c = mathutils.Vector(layout_data["Obj_C"]["loc"])

center = pos_c + (pos_b - pos_c) * 0.6
start_pos = mathutils.Vector((-1.3132, -0.6302, 4.2590))

# 计算环绕半径与初始角度
dx = start_pos.x - center.x
dy = start_pos.y - center.y
radius = math.sqrt(dx**2 + dy**2)
initial_angle = math.atan2(dy, dx)

print(f"🎯 转轴已锁定。新转轴中心: {center}, 环绕半径: {radius:.2f}米")

# 七、生成 36 0度环绕动画关键帧
for f in range(1, FRAME_COUNT + 1):
    scene.frame_set(f)
    angle_offset = 2 * math.pi * (f - 1) / FRAME_COUNT
    current_angle = initial_angle + angle_offset
    
    cam.location.x = center.x + radius * math.cos(current_angle)
    cam.location.y = center.y + radius * math.sin(current_angle)
    cam.location.z = start_pos.z 
    
    # 镜头紧紧咬死修正后的中轴中心
    direction = center - cam.location
    cam.rotation_euler = direction.to_track_quat('-Z', 'Y').to_euler()
    
    cam.keyframe_insert(data_path="location", frame=f)
    cam.keyframe_insert(data_path="rotation_euler", frame=f)

# 八、导出视频参数
scene.render.image_settings.file_format = 'FFMPEG'
scene.render.ffmpeg.format = 'MPEG4'
scene.render.ffmpeg.codec = 'H264'
scene.render.ffmpeg.constant_rate_factor = 'HIGH'

video_output = "/mnt/workspace/Layout_Colored_Orbit_v3.mp4"
scene.render.filepath = video_output

print("📺 正在使用 Blender 4.0.2 后台安全渲染【修正转轴+全色点云觉醒】视频...")
bpy.ops.render.render(animation=True)
print("🏁 最终视频渲染圆满完成！")
''')

# ================= 唤醒服务器后台 =================
blender_bin = "/mnt/workspace/blender-4.0.2-linux-x64/blender"
cmd = [blender_bin, "-b", "-P", script_path]

print("⏳ 正在修正 Blender 4.0 节点API，重新编译渲染管线...\n")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")

process.wait()

print("\n🎉 4.0 节点漏洞修复成功！视频已完成重构。")
print("📁 请去左侧刷新并下载最新成果: /mnt/workspace/Layout_Colored_Orbit_v3.mp4")

⏳ 正在修正 Blender 4.0 节点API，重新编译渲染管线...

OBJ import: cannot read from MTL file: '/mnt/workspace/data/model.mtl'
Cannot load image file: '/mnt/workspace/data/texture_kd.jpg'
Failed to open dir (No such file or directory): 
Cannot load image file: 'texture_kd.jpg'
Failed to open dir (No such file or directory): 
Cannot load image file: 'texture kd.jpg'
OBJ import: cannot read from MTL file: '/mnt/workspace/data/model.mtl'
Loaded image from: '/mnt/workspace/data/texture_kd.jpg'
Blender 4.0.2 (hash 9be62e85b727 built 2023-12-05 08:48:50)
OBJ import of 'counter_surface_mesh_poisson.obj' took 131.5 ms
OBJ import of 'obj_b.obj' took 64.6 ms
OBJ import of 'obj_c.obj' took 29.4 ms
Fra:1 Mem:47.48M (Peak 74.05M) | Time:00:00.02 | Mem:0.00M, Peak:0.00M | Scene, ViewLayer | Synchronizing object | TABLE
Fra:1 Mem:47.48M (Peak 74.05M) | Time:00:00.02 | Mem:0.00M, Peak:0.00M | Scene, ViewLayer | Synchronizing object | Obj_C
Fra:1 Mem:86.15M (Peak 116.42M) | Time:00:00.09 | Mem:0.00M, Peak:0.00M | Scene,

## FINAL

In [ ]:
import subprocess

# 升级脚本版本号至 v4，整合新资产 tissue
script_path = "/mnt/workspace/pipeline_render_colored_video_v4.py"

with open(script_path, "w", encoding="utf-8") as f:
    f.write('''import bpy
import math
import mathutils
import os

print("🚀 启动【Blender 4.0 多资产整合】36帧全彩环绕视频管线...")

# 一、初始化与彻底清空
bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete(use_global=False)

scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.device = 'CPU'
scene.cycles.samples = 12  # 保持短平快低采样，确保快速出图

scene.render.resolution_x = 1280
scene.render.resolution_y = 720

FRAME_COUNT = 36
scene.frame_start = 1
scene.frame_end = FRAME_COUNT

# 二、🛠️ 核心函数 1：强行绑定图片贴图（用于水果 B 和 C）
def force_bind_texture(obj_name, texture_path):
    obj = bpy.data.objects.get(obj_name)
    if not obj or not os.path.exists(texture_path):
        return
    mat = bpy.data.materials.new(name=f"Mat_{obj_name}")
    mat.use_nodes = True
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    bsdf = nodes.get("Principled BSDF")
    
    tex_image = nodes.new('ShaderNodeTexImage')
    tex_image.image = bpy.data.images.load(texture_path)
    links.new(tex_image.outputs['Color'], bsdf.inputs['Base Color'])
    
    if len(obj.data.materials) == 0: obj.data.materials.append(mat)
    else: obj.data.materials[0] = mat

# 三、🛠️ 核心函数 2：强行唤醒点云“顶点颜色”（用于背景 TABLE 和新加入的 tissue）
def awake_vertex_color(obj_name):
    obj = bpy.data.objects.get(obj_name)
    if not obj: return
    
    # 检查网格是否自带点云颜色属性
    if not obj.data.color_attributes:
        print(f"ℹ️ 网格 {obj_name} 未检测到嵌入的顶点颜色，将保持默认材质。")
        return
        
    active_attr = obj.data.color_attributes.active
    print(f"🎨 ✨检测到点云颜色属性: '{active_attr.name}'，正在适配 Blender 4.0 属性节点树...")
    
    mat = bpy.data.materials.new(name=f"Mat_VC_{obj_name}")
    mat.use_nodes = True
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    bsdf = nodes.get("Principled BSDF")
    
    # Blender 4.0 属性节点抓取顶点颜色
    attr_node = nodes.new('ShaderNodeAttribute')
    attr_node.attribute_name = active_attr.name
    
    # 将属性节点的 Color 输出连入原理化 BSDF 的 Base Color
    links.new(attr_node.outputs['Color'], bsdf.inputs['Base Color'])
    
    if len(obj.data.materials) == 0: obj.data.materials.append(mat)
    else: obj.data.materials[0] = mat

# 四、导入并部署资产（整合新资产 tissue 的绝对坐标、旋转和缩放）
layout_data = {
    "TABLE": {
        "path": "/mnt/workspace/data/counter_surface_mesh_poisson.obj",
        "loc": [0.0, 0.0, 0.0],
        "rot": [0.9948, 0.0, 0.0],
        "scale": [1.0, 1.0, 1.0]
    },
    "Obj_B": {
        "path": "/mnt/workspace/data/obj_b.obj",
        "tex": "/mnt/workspace/data/obj_b_text.jpg", 
        "loc": [3.0103, 0.5386, 3.1579],
        "rot": [0.1063, 0.0618, -0.1891],
        "scale": [1.0, 1.0, 1.0]
    },
    "Obj_C": {
        "path": "/mnt/workspace/data/obj_c.obj",
        "tex": "/mnt/workspace/data/obj_c_text.jpg",
        "loc": [1.0591, 0.4646, 2.4416],
        "rot": [1.5707, -1.5910, 0.0],
        "scale": [1.0, 1.0, 1.0]
    },
    "tissue": {
        "path": "/mnt/workspace/data/tissue.obj",
        "loc": [1.2476556301116943, -0.2412528097629547, 3.766942024230957],
        "rot": [6.766481399536133, -9.467084884643555, 0.6309459209442139],
        "scale": [0.5, 0.5, 0.5]
    }
}

for item_name, info in layout_data.items():
    if not os.path.exists(info["path"]): 
        print(f"⚠️ 找不到文件: {info['path']}，已跳过。")
        continue
    bpy.ops.wm.obj_import(filepath=info["path"], forward_axis='Y', up_axis='Z')
    obj = bpy.context.selected_objects[0]
    obj.name = item_name
    obj.location = info["loc"]
    obj.rotation_euler = info["rot"]
    obj.scale = info["scale"]
    
    # 分流处理材质
    if item_name in ["TABLE", "tissue"]:
        awake_vertex_color(item_name) # 点云重构生成的模型统一激活顶点真彩色
    elif "tex" in info:
        force_bind_texture(item_name, info["tex"]) # 水果物体硬绑定贴图

# 五、环境打光
bpy.ops.object.light_add(type='SUN', location=(-5, 5, 12))
sun = bpy.context.object
sun.data.energy = 6.0

# 六、创建并配置摄像机
cam_data = bpy.data.cameras.new("cam")
cam_data.lens = 20  
cam = bpy.data.objects.new("cam", cam_data)
bpy.context.collection.objects.link(cam)
scene.camera = cam

# 🎯 动态几何中心转轴策略：计算 Obj_B, Obj_C, 以及新加入的 tissue 三者的物理中心
pos_b = mathutils.Vector(layout_data["Obj_B"]["loc"])
pos_c = mathutils.Vector(layout_data["Obj_C"]["loc"])
pos_tissue = mathutils.Vector(layout_data["tissue"]["loc"])

# 三维空间三点求中心均值，使旋转轴能够照顾到所有视点物体
center = (pos_b + pos_c + pos_tissue) / 3.0
start_pos = mathutils.Vector((-1.3132, -0.6302, 4.2590))

# 计算基于新复合中心的环绕半径与初始角度
dx = start_pos.x - center.x
dy = start_pos.y - center.y
radius = math.sqrt(dx**2 + dy**2)
initial_angle = math.atan2(dy, dx)

print(f"🎯 复合平衡转轴已锁定。新转轴中心: {center}, 环绕半径: {radius:.2f}米")

# 七、生成 360 度多资产环绕关键帧
for f in range(1, FRAME_COUNT + 1):
    scene.frame_set(f)
    angle_offset = 2 * math.pi * (f - 1) / FRAME_COUNT
    current_angle = initial_angle + angle_offset
    
    cam.location.x = center.x + radius * math.cos(current_angle)
    cam.location.y = center.y + radius * math.sin(current_angle)
    cam.location.z = start_pos.z 
    
    # 镜头时刻锁死新加入的复合中心点
    direction = center - cam.location
    cam.rotation_euler = direction.to_track_quat('-Z', 'Y').to_euler()
    
    cam.keyframe_insert(data_path="location", frame=f)
    cam.keyframe_insert(data_path="rotation_euler", frame=f)

# 八、导出视频参数
scene.render.image_settings.file_format = 'FFMPEG'
scene.render.ffmpeg.format = 'MPEG4'
scene.render.ffmpeg.codec = 'H264'
scene.render.ffmpeg.constant_rate_factor = 'HIGH'

video_output = "/mnt/workspace/Layout_Colored_Orbit_v4.mp4"
scene.render.filepath = video_output

print("📺 正在使用 Blender 4.0.2 后台渲染【包含新资产 Tissue】的无缝彩色环绕视频...")
bpy.ops.render.render(animation=True)
print("🏁 最终多资产整合视频渲染圆满完成！")
''')

# ================= 唤醒服务器后台 =================
blender_bin = "/mnt/workspace/blender-4.0.2-linux-x64/blender"
cmd = [blender_bin, "-b", "-P", script_path]

print("⏳ 正在注入 Tissue 位姿数据，重新生成多物体三维舞台渲染管线...\n")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")

process.wait()

print("\n🎉 多资产舞台重构成功！")
print("📁 请去左侧刷新并下载最新的整合版成果: /mnt/workspace/Layout_Colored_Orbit_v4.mp4")

⏳ 正在注入 Tissue 位姿数据，重新生成多物体三维舞台渲染管线...

OBJ import: cannot read from MTL file: '/mnt/workspace/data/model.mtl'
Cannot load image file: '/mnt/workspace/data/texture_kd.jpg'
Failed to open dir (No such file or directory): 
Cannot load image file: 'texture_kd.jpg'
Failed to open dir (No such file or directory): 
Cannot load image file: 'texture kd.jpg'
OBJ import: cannot read from MTL file: '/mnt/workspace/data/model.mtl'
Loaded image from: '/mnt/workspace/data/texture_kd.jpg'
Blender 4.0.2 (hash 9be62e85b727 built 2023-12-05 08:48:50)
OBJ import of 'counter_surface_mesh_poisson.obj' took 264.7 ms
OBJ import of 'obj_b.obj' took 182.4 ms
OBJ import of 'obj_c.obj' took 156.2 ms
OBJ import of 'tissue.obj' took 80.6 ms
Fra:1 Mem:61.49M (Peak 94.83M) | Time:00:00.07 | Mem:0.00M, Peak:0.00M | Scene, ViewLayer | Synchronizing object | TABLE
Fra:1 Mem:61.49M (Peak 94.83M) | Time:00:00.07 | Mem:0.00M, Peak:0.00M | Scene, ViewLayer | Synchronizing object | tissue
Fra:1 Mem:118.14M (Peak 169.31M) | T

In [ ]:
import subprocess

# 升级脚本版本号至 v5，打造慢速+放大+巨幅缩小的电影级运镜
script_path = "/mnt/workspace/pipeline_render_colored_video_v5.py"

with open(script_path, "w", encoding="utf-8") as f:
    f.write('''import bpy
import math
import mathutils
import os

print("🚀 启动【Blender 4.0 电影级运镜】150帧多段全彩环绕视频管线...")

# 一、初始化与彻底清空
bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete(use_global=False)

scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.device = 'CPU'
scene.cycles.samples = 12  # 低采样确保多帧数下依然能快速出图

scene.render.resolution_x = 1280
scene.render.resolution_y = 720

# 🎯 核心改动 1：总帧数提升至 150 帧，让旋转速度明显慢下来，同时给推拉镜头留足空间
FRAME_COUNT = 150
scene.frame_start = 1
scene.frame_end = FRAME_COUNT

# 二、🛠️ 材质绑定核心函数
def force_bind_texture(obj_name, texture_path):
    obj = bpy.data.objects.get(obj_name)
    if not obj or not os.path.exists(texture_path): return
    mat = bpy.data.materials.new(name=f"Mat_{obj_name}")
    mat.use_nodes = True
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    bsdf = nodes.get("Principled BSDF")
    tex_image = nodes.new('ShaderNodeTexImage')
    tex_image.image = bpy.data.images.load(texture_path)
    links.new(tex_image.outputs['Color'], bsdf.inputs['Base Color'])
    if len(obj.data.materials) == 0: obj.data.materials.append(mat)
    else: obj.data.materials[0] = mat

def awake_vertex_color(obj_name):
    obj = bpy.data.objects.get(obj_name)
    if not obj or not obj.data.color_attributes: return
    active_attr = obj.data.color_attributes.active
    mat = bpy.data.materials.new(name=f"Mat_VC_{obj_name}")
    mat.use_nodes = True
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    bsdf = nodes.get("Principled BSDF")
    attr_node = nodes.new('ShaderNodeAttribute')
    attr_node.attribute_name = active_attr.name
    links.new(attr_node.outputs['Color'], bsdf.inputs['Base Color'])
    if len(obj.data.materials) == 0: obj.data.materials.append(mat)
    else: obj.data.materials[0] = mat

# 三、资产精准部署（采用您最新的 tissue 坐标数据）
layout_data = {
    "TABLE": {
        "path": "/mnt/workspace/data/counter_surface_mesh_poisson.obj",
        "loc": [0.0, 0.0, 0.0], "rot": [0.9948, 0.0, 0.0], "scale": [1.0, 1.0, 1.0]
    },
    "Obj_B": {
        "path": "/mnt/workspace/data/obj_b.obj", "tex": "/mnt/workspace/data/obj_b_text.jpg", 
        "loc": [3.0103, 0.5386, 3.1579], "rot": [0.1063, 0.0618, -0.1891], "scale": [1.0, 1.0, 1.0]
    },
    "Obj_C": {
        "path": "/mnt/workspace/data/obj_c.obj", "tex": "/mnt/workspace/data/obj_c_text.jpg",
        "loc": [1.0591, 0.4646, 2.4416], "rot": [1.5707, -1.5910, 0.0], "scale": [1.0, 1.0, 1.0]
    },
    "tissue": {
        "path": "/mnt/workspace/data/tissue.obj",
        "loc": [1.2476556301116943, -0.2412528097629547, 3.766942024230957],
        "rot": [6.766481399536133, -9.467084884643555, 0.6309459209442139],
        "scale": [0.5, 0.5, 0.5]
    }
}

for item_name, info in layout_data.items():
    if not os.path.exists(info["path"]): continue
    bpy.ops.wm.obj_import(filepath=info["path"], forward_axis='Y', up_axis='Z')
    obj = bpy.context.selected_objects[0]
    obj.name = item_name
    obj.location = info["loc"]
    obj.rotation_euler = info["rot"]
    obj.scale = info["scale"]
    
    if item_name in ["TABLE", "tissue"]: awake_vertex_color(item_name)
    elif "tex" in info: force_bind_texture(item_name, info["tex"])

# 四、灯光与摄像机基础配置
bpy.ops.object.light_add(type='SUN', location=(-5, 5, 12))
bpy.context.object.data.energy = 6.0

cam_data = bpy.data.cameras.new("cam")
cam_data.lens = 20  
cam = bpy.data.objects.new("cam", cam_data)
bpy.context.collection.objects.link(cam)
scene.camera = cam

# 计算中心转轴与基础几何尺寸
pos_b = mathutils.Vector(layout_data["Obj_B"]["loc"])
pos_c = mathutils.Vector(layout_data["Obj_C"]["loc"])
pos_tissue = mathutils.Vector(layout_data["tissue"]["loc"])
center = (pos_b + pos_c + pos_tissue) / 3.0
start_pos = mathutils.Vector((-1.3132, -0.6302, 4.2590))

dx = start_pos.x - center.x
dy = start_pos.y - center.y
base_radius = math.sqrt(dx**2 + dy**2)
initial_angle = math.atan2(dy, dx)

# 五、🎯 核心改动 2：多段高级运动轨迹算法（慢速环绕 + 丝滑推拉镜头）
print("🎬 开始精细化编排 150 帧多阶摄像机动效关键帧...")
for f in range(1, FRAME_COUNT + 1):
    scene.frame_set(f)
    
    # 1. 慢速平滑旋转：150帧内均匀旋转 1.25 圈，彻底告别以前快转的感觉
    angle_offset = 2.5 * math.pi * (f - 1) / FRAME_COUNT
    current_angle = initial_angle + angle_offset
    
    # 2. 余弦平滑插值算法：杜绝镜头卡顿，实现完美的推拉速度渐变
    if f <= 60:
        # 第一阶段 (1~60帧)：从 1.0 倍距离平滑缩小(放大看细节)到 0.4 倍特写距离
        t = (f - 1) / 59.0
        radius_factor = 1.0 - 0.6 * (0.5 - 0.5 * math.cos(t * math.pi))
    else:
        # 第二阶段 (61~150帧)：从 0.4 倍特写距离暴增拉远到 3.5 倍巨远距离（缩小到很远）
        t = (f - 60) / 90.0
        radius_factor = 0.4 + 3.1 * (0.5 - 0.5 * math.cos(t * math.pi))
        
    # 计算当前帧对应的实际缩放半径
    current_radius = base_radius * radius_factor
    
    # 计算摄像机在当前帧的三维空间位置
    cam.location.x = center.x + current_radius * math.cos(current_angle)
    cam.location.y = center.y + current_radius * math.sin(current_angle)
    
    # 3. 智能无人机高度协同：拉远时镜头同步升高（因 factor 最大可达 3.5），转为大气恢弘的俯瞰视角
    cam.location.z = start_pos.z * (0.6 + 0.4 * radius_factor)
    
    # 4. 强行锁定：无论拉多远，镜头中轴线死死咬住三个资产的物理中心
    direction = center - cam.location
    cam.rotation_euler = direction.to_track_quat('-Z', 'Y').to_euler()
    
    # 记录当前帧的关键帧数据
    cam.keyframe_insert(data_path="location", frame=f)
    cam.keyframe_insert(data_path="rotation_euler", frame=f)

# 六、视频高质量渲染导出
scene.render.image_settings.file_format = 'FFMPEG'
scene.render.ffmpeg.format = 'MPEG4'
scene.render.ffmpeg.codec = 'H264'
scene.render.ffmpeg.constant_rate_factor = 'HIGH'

video_output = "/mnt/workspace/Layout_Colored_Orbit_v5.mp4"
scene.render.filepath = video_output

print("📺 正在调用后台 Cycles 引擎，渲染全新高级镜头（慢速 + 先放大特写 + 后极远拉伸）视频...")
bpy.ops.render.render(animation=True)
print("🏁 高级镜头视频管线渲染圆满落幕！")
''')

# ================= 唤醒服务器后台 =================
blender_bin = "/mnt/workspace/blender-4.0.2-linux-x64/blender"
cmd = [blender_bin, "-b", "-P", script_path]

print("⏳ 正在编译电影级多阶插值算法，重构三维运动相机轨迹...\n")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")

process.wait()

print("\n🎉 电影级高级运镜视频重构成功！")
print("📁 请去左侧刷新并下载最新震撼成果: /mnt/workspace/Layout_Colored_Orbit_v5.mp4")

⏳ 正在编译电影级多阶插值算法，重构三维运动相机轨迹...

OBJ import: cannot read from MTL file: '/mnt/workspace/data/model.mtl'
Cannot load image file: '/mnt/workspace/data/texture_kd.jpg'
Failed to open dir (No such file or directory): 
Cannot load image file: 'texture_kd.jpg'
Failed to open dir (No such file or directory): 
Cannot load image file: 'texture kd.jpg'
OBJ import: cannot read from MTL file: '/mnt/workspace/data/model.mtl'
Loaded image from: '/mnt/workspace/data/texture_kd.jpg'
Blender 4.0.2 (hash 9be62e85b727 built 2023-12-05 08:48:50)
OBJ import of 'counter_surface_mesh_poisson.obj' took 117.2 ms
OBJ import of 'obj_b.obj' took 35.2 ms
OBJ import of 'obj_c.obj' took 28.3 ms
OBJ import of 'tissue.obj' took 79.8 ms
Fra:1 Mem:61.86M (Peak 95.37M) | Time:00:00.02 | Mem:0.00M, Peak:0.00M | Scene, ViewLayer | Synchronizing object | TABLE
Fra:1 Mem:61.86M (Peak 95.37M) | Time:00:00.02 | Mem:0.00M, Peak:0.00M | Scene, ViewLayer | Synchronizing object | Obj_C
Fra:1 Mem:61.86M (Peak 95.37M) | Time:00:00.02 